# ReDifraX — Pipeline de Análise XRD

Automatiza a identificação e quantificação de fases cristalinas em difratogramas de pó (formato RRUFF).

| Etapa | Módulo | O que faz |
|-------|--------|-----------|
| 1. Configuração | `config.py` | Define caminhos do projeto via `ProjectConfig` |
| 2. Download | `Diretories_Downloads.py` | Baixa CIFs do Crystallography Open Database (COD) |
| 3. Filtro primário | `Primary_Filter.py` | Simula padrões XRD com pymatgen e rankeia por correlação de Pearson |
| 4. Refinamento | `Workflow_GSAS2.py` | Refinamento de Rietveld passo a passo via GSAS-II |
| 5. Resultados | `Extraction_Phases.py` | Extrai frações de fase e de peso do arquivo `.lst` |

---

In [ ]:
import logging

from functions.config import ProjectConfig
from functions.Diretories_Downloads import cif_download
from functions.Primary_Filter import primary_filter
from functions.Extraction_Phases import extrair_fracoes_fase
from functions.Plots import plot_filtro_primario, plot_rietveld, plot_refinamento_com_referencias

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

## 1. Configuração

In [ ]:
cfg = ProjectConfig("projeto_x")
cfg.create_dirs()

print(cfg.project_dir)
print(cfg.ref_dir)
print(cfg.results_dir)

## 2. Download das Estruturas de Referência (COD)

Edite o dicionário abaixo para adicionar ou remover fases. Os IDs numéricos são do [Crystallography Open Database](http://www.crystallography.net/cod/).

In [ ]:
# CELL 3: Download referências

dict_refs = {
    # alpha-Fe2O3: Trigonal, R-3c. Referência clássica de alta cristalinidade.
    "Hematite": 2101168,

    # Fe3O4: Cúbico, Fd-3m. Estrutura de espinélio invertido.
    # Cuidado: Picos muito próximos aos da Maghemita.
    "Magnetite": 9005813,

    # gamma-Fe2O3: Cúbico, P4332. Fase metaestável.
    # Importante conferir se não há picos de superestrutura no seu DRX.
    "Maghemite": 9017489,

    # alpha-FeOOH: Ortorrômbico, Pnma. Comum se houver hidratação no resíduo.
    "Goethite": 7720752,

    # FeO: Cúbico, Fm-3m. FASE CRÍTICA para a redução a 850 °C.
    "Wüstite": 9002673,

    # Fe3C: Ortorrômbico, Pnma. Fase crítica para a redução a 850 °C.
    "Cementite": 9016584,

    # alpha-Fe: Cúbico, Im-3m. O produto final da redução (Ferro metálico).
    "Fe_metalico": 9008536,
}

for Nome, cod_ID in dict_refs.items():
    cif_download(Nome, cod_ID, str(cfg.ref_dir))

## 3. Filtro Primário

Compara a amostra experimental com cada CIF baixado e retorna um ranking por correlação de Pearson.  
Ajuste `correlation="cosine"` se preferir similaridade por cosseno.

In [ ]:
input_drx_raw = "inputs/hematite_raw.txt"
df, amostra_norm, theta = primary_filter(
    correlation="pearson",
    input_file=input_drx_raw,
    ref_dir=str(cfg.ref_dir),
)
print(df[["nome", "score"]])

In [ ]:
plot_filtro_primario(theta, amostra_norm, df)

## 4. Refinamento de Rietveld (GSAS-II)

Seleciona as `N` melhores fases do filtro primário e executa o refinamento sequencial em 5 passos:  
Escala → Deslocamento → Célula → Perfil → Posições atômicas.

> **Dica:** use `refinar_atomos=False` se o wR ficar alto ou o refinamento divergir.

In [ ]:
import importlib, functions.Workflow_GSAS2
importlib.reload(functions.Workflow_GSAS2)
from functions.Workflow_GSAS2 import refinamento_sequencial_oxidos

refs_cif = [
    str(cfg.ref_dir / df["nome"].iloc[0]),
    str(cfg.ref_dir / df["nome"].iloc[1]),
    # str(cfg.ref_dir / df["nome"].iloc[2]),
]

input_inst = "inputs/inst_rruff.prm"
resultado = refinamento_sequencial_oxidos(
    input_drx_raw,
    input_inst,
    refs_cif,
    cfg.project_name,
    refinar_atomos=True,  # use False para XRD de baixa qualidade
)

In [ ]:
plot_rietveld(resultado)

In [ ]:
plot_refinamento_com_referencias(
    resultado["x"],
    resultado["yobs"],
    resultado["ycalc"],
    refs_cif,
    theta,
)

## 5. Resultados — Frações de Fase

Extrai as frações diretamente do arquivo `.lst` gerado pelo GSAS-II.  
`Weight Fraction` representa a porcentagem em massa de cada fase na amostra.

In [ ]:
lst_file_path = str(cfg.results_dir / f"{cfg.project_name}.lst")
dados_extraidos = extrair_fracoes_fase(lst_file_path)

print("--- Resultados do Refinamento ---")
if isinstance(dados_extraidos, dict):
    for fase, fracoes in dados_extraidos.items():
        pf  = fracoes.get("Phase Fraction")
        wf  = fracoes.get("Weight Fraction")
        pf_str = f"{pf:.6f}" if pf is not None else "N/A"
        wf_str = f"{wf:.4f}  ({wf * 100:.2f}%)" if wf is not None else "N/A"
        print(f"\nFase: {fase}")
        print(f"  Phase Fraction  : {pf_str}")
        print(f"  Weight Fraction : {wf_str}")
else:
    print(dados_extraidos)